# RSNA input audit

This notebook contains the reviewed Python audit sources and uses the standard library only. Attach competition inputs, keep Internet Off, and Run All. No CLI authentication or package installation is needed.

The default run discovers input structure with bounded traversal. Generate a CSV-inspection variant locally with `python scripts/build_audit_notebook.py --inspect-csv --output notebooks/kaggle_audit_csv.ipynb`; import that complete file instead of adding cells. This notebook does not decode DICOMs or train models. Synthetic checks do not verify competition access, actual data, or model performance.


In [ ]:
# Reviewed source bundle: no filesystem extraction or network access.
import hashlib
import importlib.abc
import importlib.util
import json
import sys

_RSNA_BUNDLE = json.loads('{"sha256":{"__init__.py":"3f6041d85fd46b145dd12c9989a821f89803370d4ff60b49ed6274880181b44f","audit.py":"5931b8daa1ba89014c0b4ccfd967b2f5887604707afefd61aa145027ff88eed9","cli.py":"c539fbcea21c511e09c2563aee48229ef57439c33672b835e12160354ef0d0ce","config.py":"827813759e6d8b9fab22aa5de73371926bfe8affa0b68eb623954c3e840e6314","contracts.py":"b034b092077134c124a3832bcb4781d5425553cfd4ba2d741cd7fa00cafbd1d1","inputs.py":"4a9ff8704400fc93ca42f8d82b22aff8eba270f94e62295b4dc95239cf777230","metrics.py":"6932cb559c58c055d3a333b41cf9f894e0a1458c3f47abb4467536670fe8c496","submission.py":"8a730b7dccfff385fb4c0b62270dbec4a881334808a07203d2cbc72dcc1cc6ef"},"sources":{"__init__":"\\"\\"\\"Initial scaffold only; data validity and model performance are unmeasured.\\"\\"\\"\\n\\n__version__ = \\"0.1.0\\"\\n","audit":"\\"\\"\\"Initial preflight contracts only; never a Phase B or training approval.\\"\\"\\"\\n\\nfrom datetime import datetime, timezone\\nimport json\\nfrom pathlib import Path\\nimport platform\\nimport time\\nimport uuid\\n\\nfrom .config import AuditConfig\\nfrom .contracts import ContractError, TARGETS, audit_series, audit_studies\\nfrom .inputs import InputError, MANIFEST_NAMES, discover_inputs, read_csv_manifest, safe_absolute\\nfrom .submission import validate_submission\\n\\n\\ndef _summarize(contract: dict) -> dict:\\n    # Keep outputs compact; no full identifier tables, report text, or pixels.\\n    return {key: value for key, value in contract.items() if key not in (\\"study_ids\\", \\"series_ids\\")}\\n\\n\\ndef _csv_contracts(root: str, config: AuditConfig, progress: dict) -> dict:\\n    tables = {}\\n    columns = {}\\n    hashes = {}\\n    for name in MANIFEST_NAMES:\\n        progress[\\"attempted_files\\"].append(name)\\n        columns[name], tables[name], hashes[name] = read_csv_manifest(Path(root) / name, config)\\n        progress[\\"parsed_files\\"].append(name)\\n        progress[\\"sha256\\"][name] = hashes[name]\\n    train = audit_studies(tables[\\"train.csv\\"], \\"train\\")\\n    test = audit_studies(tables[\\"test.csv\\"], \\"test\\")\\n    train_ids, test_ids = set(train[\\"study_ids\\"]), set(test[\\"study_ids\\"])\\n    overlap = train_ids & test_ids\\n    if overlap:\\n        raise ContractError(\\"Train/test study identifiers overlap\\")\\n    train_series = audit_series(tables[\\"train_series.csv\\"], train_ids)\\n    test_series = audit_series(tables[\\"test_series.csv\\"], test_ids)\\n    series_overlap = set(train_series[\\"series_ids\\"]) & set(test_series[\\"series_ids\\"])\\n    if series_overlap:\\n        raise ContractError(\\"Train/test series identifiers overlap\\")\\n    missing_train = train_ids - {row[\\"StudyInstanceUID\\"] for row in tables[\\"train_series.csv\\"]}\\n    missing_test = test_ids - {row[\\"StudyInstanceUID\\"] for row in tables[\\"test_series.csv\\"]}\\n    if missing_train or missing_test:\\n        raise ContractError(\\"Study manifest includes studies without a series reference\\")\\n    sample = []\\n    for row in tables[\\"sample_submission.csv\\"]:\\n        converted = dict(row)\\n        for target in TARGETS:\\n            if target not in converted:\\n                raise ContractError(\\"Sample submission is missing target columns\\")\\n            try:\\n                converted[target] = float(converted[target])\\n            except (TypeError, ValueError) as error:\\n                raise ContractError(\\"Sample submission scores must be numeric\\") from error\\n        sample.append(converted)\\n    submission = validate_submission(sample, test_ids, columns=columns[\\"sample_submission.csv\\"])\\n    valid = all(result[\\"status\\"] == \\"PASS\\" for result in (train, test, train_series, test_series))\\n    return {\\n        \\"status\\": \\"VALID\\" if valid else \\"INVALID\\", \\"columns\\": columns,\\n        \\"sha256\\": hashes, \\"train\\": _summarize(train), \\"test\\": _summarize(test),\\n        \\"train_series\\": _summarize(train_series), \\"test_series\\": _summarize(test_series),\\n        \\"sample_submission\\": submission, \\"train_test_study_overlap_count\\": 0,\\n        \\"train_test_series_overlap_count\\": 0,\\n        \\"scope\\": \\"CSV contracts only; no patient, report-quality, geometry or image-integrity audit\\",\\n    }\\n\\n\\ndef run_preflight(config: AuditConfig, source_identity: dict | None = None) -> dict:\\n    if not isinstance(config, AuditConfig):\\n        raise TypeError(\\"run_preflight requires validated AuditConfig\\")\\n    started = time.perf_counter()\\n    discovery = discover_inputs(config)\\n    csv_contracts = {\\"status\\": \\"NOT_REQUESTED\\"}\\n    status = discovery[\\"status\\"]\\n    if config.inspect_csv:\\n        csv_contracts = {\\"status\\": \\"BLOCKED_BY_DISCOVERY\\"}\\n        if discovery[\\"selected_root\\"]:\\n            progress = {\\"attempted_files\\": [], \\"parsed_files\\": [], \\"sha256\\": {}}\\n            discovery[\\"csv_read_attempted\\"] = True\\n            discovery[\\"csv_read_completed\\"] = False\\n            discovery[\\"csv_preflight_completed\\"] = False\\n            try:\\n                csv_contracts = _csv_contracts(discovery[\\"selected_root\\"], config, progress)\\n                status = \\"CONTRACTS_VALID\\" if csv_contracts[\\"status\\"] == \\"VALID\\" else \\"CONTRACTS_INVALID\\"\\n                discovery[\\"csv_preflight_completed\\"] = True\\n            except (InputError, ContractError, OSError) as error:\\n                # Exceptions contain only static contract descriptions; no raw reports.\\n                csv_contracts = {\\"status\\": \\"INVALID\\", \\"error_type\\": type(error).__name__,\\n                                 \\"reason\\": str(error) if isinstance(error, (InputError, ContractError)) else \\"Filesystem read failed\\"}\\n                status = \\"CONTRACTS_INVALID\\"\\n            discovery[\\"csv_read_completed\\"] = len(progress[\\"parsed_files\\"]) == len(MANIFEST_NAMES)\\n            # A parse failure may still have consumed bytes. None means unknown,\\n            # while successful parsed files establish that contents were read.\\n            discovery[\\"csv_contents_read\\"] = True if progress[\\"parsed_files\\"] else None\\n            discovery[\\"csv_read_progress\\"] = progress\\n    return {\\n        \\"schema\\": \\"rsna-knee-preflight-v1\\", \\"phase\\": \\"INITIAL_SCAFFOLD\\", \\"status\\": status,\\n        \\"created_utc\\": datetime.now(timezone.utc).isoformat(), \\"configuration\\": config.to_dict(),\\n        \\"source_identity\\": source_identity or {\\"status\\": \\"NOT_SUPPLIED\\"},\\n        \\"runtime\\": {\\"python\\": platform.python_version(), \\"platform\\": platform.system(), \\"gpu_used\\": False},\\n        \\"discovery\\": discovery, \\"csv_contracts\\": csv_contracts,\\n        \\"phase_b_status\\": \\"NOT_RUN\\", \\"training_ready\\": False, \\"phase1_pass\\": False,\\n        \\"gold_resource\\": {\\"status\\": \\"NOT_ESTABLISHED\\", \\"observed_study_count\\": None},\\n        \\"model_metrics\\": None,\\n        \\"unassessed\\": [\\"patient grouping\\", \\"report quality and languages\\", \\"DICOM UID inventory\\",\\n                       \\"physical geometry\\", \\"transfer syntaxes and decoding\\", \\"gold provenance\\",\\n                       \\"validation split viability\\", \\"Kaggle account access\\", \\"model and inference runtime\\"],\\n        \\"duration_before_report_write_seconds\\": time.perf_counter() - started,\\n    }\\n\\n\\ndef write_report(report: dict, config: AuditConfig) -> Path:\\n    output = safe_absolute(config.output_dir)\\n    source = safe_absolute(config.input_base)\\n    if output.is_relative_to(source):\\n        raise InputError(\\"Report destination must be outside input_base\\")\\n    payload = json.dumps(report, indent=2, sort_keys=True, allow_nan=False) + \\"\\\\n\\"\\n    if len(payload.encode(\\"utf-8\\")) > config.max_report_bytes:\\n        raise InputError(\\"REPORT_SIZE_LIMIT_EXCEEDED\\")\\n    output.mkdir(parents=True, exist_ok=True)\\n    stamp = datetime.now(timezone.utc).strftime(\\"%Y%m%dT%H%M%S%fZ\\")\\n    path = output / f\\"preflight_{stamp}_{uuid.uuid4().hex[:8]}.json\\"\\n    with path.open(\\"x\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as stream:\\n        stream.write(payload)\\n    return path\\n","cli":"\\"\\"\\"Local entry point with no authenticated services or training commands.\\"\\"\\"\\n\\nimport argparse\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport sys\\n\\nfrom .audit import run_preflight, write_report\\nfrom .config import AuditConfig, ConfigError, load_config\\nfrom .inputs import InputError\\n\\n\\ndef _source_identity() -> dict:\\n    package = Path(__file__).parent\\n    names = (\\"__init__\\", \\"config\\", \\"inputs\\", \\"contracts\\", \\"metrics\\", \\"submission\\", \\"audit\\", \\"cli\\")\\n    return {\\"modules\\": {name: hashlib.sha256((package / (name + \\".py\\")).read_bytes()).hexdigest() for name in names}}\\n\\n\\ndef main(argv=None) -> int:\\n    parser = argparse.ArgumentParser(description=\\"Read-only RSNA input/CSV preflight; no Phase B execution or training\\")\\n    parser.add_argument(\\"--config\\", type=Path)\\n    parser.add_argument(\\"--input-base\\")\\n    parser.add_argument(\\"--data-root\\")\\n    parser.add_argument(\\"--output-dir\\")\\n    parser.add_argument(\\"--inspect-csv\\", action=\\"store_true\\", default=None,\\n                        help=\\"Check bounded CSV contracts; source data is not modified\\")\\n    args = parser.parse_args(argv)\\n    try:\\n        config = load_config(args.config) if args.config else AuditConfig()\\n        overrides = {name: getattr(args, name) for name in (\\"input_base\\", \\"data_root\\", \\"output_dir\\", \\"inspect_csv\\") if getattr(args, name) is not None}\\n        config = config.with_overrides(**overrides)\\n        report = run_preflight(config, source_identity=_source_identity())\\n        path = write_report(report, config)\\n    except (ConfigError, InputError, OSError, ValueError) as error:\\n        message = str(error) if isinstance(error, (ConfigError, InputError)) else \\"Preflight failed; check configuration and filesystem permissions\\"\\n        print(json.dumps({\\"status\\": \\"ERROR\\", \\"error_type\\": type(error).__name__, \\"reason\\": message}), file=sys.stderr)\\n        return 2\\n    print(json.dumps({\\"status\\": report[\\"status\\"], \\"report_path\\": str(path),\\n                      \\"phase_b_status\\": report[\\"phase_b_status\\"], \\"training_ready\\": report[\\"training_ready\\"]}, indent=2))\\n    return 0 if report[\\"status\\"] in (\\"MANIFESTS_LOCATED\\", \\"CONTRACTS_VALID\\") else 2\\n","config":"\\"\\"\\"Small immutable configuration; no credential or environment discovery.\\"\\"\\"\\n\\nfrom dataclasses import asdict, dataclass, fields, replace\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport stat\\nfrom typing import Mapping\\n\\n\\nclass ConfigError(ValueError):\\n    \\"\\"\\"Configuration cannot safely represent the requested preflight.\\"\\"\\"\\n\\n\\n@dataclass(frozen=True)\\nclass AuditConfig:\\n    input_base: str = \\"data\\"\\n    output_dir: str = \\"artifacts\\"\\n    data_root: str | None = None\\n    inspect_csv: bool = False\\n    max_depth: int = 2\\n    max_directories: int = 128\\n    max_entries: int = 2048\\n    max_file_bytes: int = 8 * 1024 * 1024\\n    max_csv_rows: int = 100000\\n    max_report_bytes: int = 1024 * 1024\\n\\n    def __post_init__(self):\\n        for name in (\\"input_base\\", \\"output_dir\\", \\"data_root\\"):\\n            value = getattr(self, name)\\n            if value is None and name == \\"data_root\\":\\n                continue\\n            if not isinstance(value, str) or not value.strip() or value != value.strip():\\n                raise ConfigError(f\\"{name} must be a nonempty unpadded path string\\")\\n            if any(ord(character) < 32 for character in value):\\n                raise ConfigError(f\\"{name} contains a control character\\")\\n        if type(self.inspect_csv) is not bool:\\n            raise ConfigError(\\"inspect_csv must be a JSON boolean\\")\\n        limits = {\\n            \\"max_depth\\": (0, 4), \\"max_directories\\": (1, 1024),\\n            \\"max_entries\\": (1, 65536), \\"max_file_bytes\\": (1, 64 * 1024 * 1024),\\n            \\"max_csv_rows\\": (1, 1000000), \\"max_report_bytes\\": (1, 4 * 1024 * 1024),\\n        }\\n        for name, (lower, upper) in limits.items():\\n            value = getattr(self, name)\\n            if type(value) is not int or not lower <= value <= upper:\\n                raise ConfigError(f\\"{name} must be an integer in [{lower}, {upper}]\\")\\n\\n    @classmethod\\n    def from_dict(cls, values: Mapping):\\n        if not isinstance(values, Mapping):\\n            raise ConfigError(\\"Configuration must be a JSON object\\")\\n        unknown = set(values) - {field.name for field in fields(cls)}\\n        if unknown:\\n            raise ConfigError(\\"Unknown configuration fields: \\" + \\", \\".join(sorted(str(key) for key in unknown)))\\n        return cls(**dict(values))\\n\\n    def to_dict(self) -> dict:\\n        return asdict(self)\\n\\n    def with_overrides(self, **values):\\n        try:\\n            return replace(self, **values)\\n        except TypeError as error:\\n            raise ConfigError(\\"Invalid configuration override\\") from error\\n\\n\\ndef load_config(path: str | Path) -> AuditConfig:\\n    def unique_pairs(pairs):\\n        values = {}\\n        for key, value in pairs:\\n            if key in values:\\n                raise ConfigError(f\\"Duplicate configuration field: {key}\\")\\n            values[key] = value\\n        return values\\n\\n    source = Path(os.path.abspath(path))\\n    protected = {\\".git\\", \\".aws\\", \\".kaggle\\", \\".codex\\", \\"credentials\\", \\"access_token\\", \\"kaggle.json\\"}\\n    if source.suffix.casefold() != \\".json\\" or any(part.casefold() in protected or part.casefold().startswith(\\".env\\") for part in source.parts):\\n        raise ConfigError(\\"Configuration must be a project JSON file outside protected paths\\")\\n    for ancestor in (source, *source.parents):\\n        try:\\n            info = ancestor.lstat()\\n        except FileNotFoundError:\\n            continue\\n        if stat.S_ISLNK(info.st_mode) or getattr(info, \\"st_file_attributes\\", 0) & 0x400:\\n            raise ConfigError(\\"Configuration cannot use symbolic links or reparse paths\\")\\n    if not stat.S_ISREG(source.lstat().st_mode):\\n        raise ConfigError(\\"Configuration must be a regular project JSON file\\")\\n    with source.open(\\"rb\\") as stream:\\n        raw = stream.read(64 * 1024 + 1)\\n    if len(raw) > 64 * 1024:\\n        raise ConfigError(\\"Configuration exceeds the 64 KiB budget\\")\\n    try:\\n        values = json.loads(raw.decode(\\"utf-8-sig\\"), object_pairs_hook=unique_pairs)\\n    except (json.JSONDecodeError, UnicodeError) as error:\\n        raise ConfigError(\\"Configuration must be valid UTF-8 JSON\\") from error\\n    return AuditConfig.from_dict(values)\\n","contracts":"\\"\\"\\"Small, explicit contracts for competition identifiers and supervision.\\n\\nUnknown labels never become negatives. These functions inspect tabular values;\\nthey neither decode images nor infer labels from reports.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nimport unicodedata\\nfrom collections import Counter\\nfrom collections.abc import Mapping\\nfrom decimal import Decimal\\nfrom numbers import Real\\nfrom typing import Any\\n\\nTARGETS = (\\n    \\"ACL\\", \\"MCL\\", \\"Medial Meniscus\\", \\"Lateral Meniscus\\", \\"Medial OA\\",\\n    \\"Lateral OA\\", \\"PF OA\\", \\"Effusion\\", \\"Synovitis\\", \\"Baker\'s\\",\\n    \\"Contusion\\", \\"Fracture\\",\\n)\\nSTUDY_ID = \\"StudyInstanceUID\\"\\nSERIES_ID = \\"SeriesInstanceUID\\"\\nSERIES_COLUMNS = (\\n    STUDY_ID, SERIES_ID, \\"Anatomical_Plane\\", \\"Fluid_Sensitive\\", \\"Fat_Suppression\\",\\n)\\n\\n\\nclass ContractError(ValueError):\\n    \\"\\"\\"A table violates a declared data contract.\\"\\"\\"\\n\\n\\ndef validate_identifier(value: str) -> str:\\n    \\"\\"\\"Return an unchanged safe identifier without imposing a numeric UID format.\\"\\"\\"\\n    if not isinstance(value, str) or not value or value != value.strip():\\n        raise ContractError(\\"identifier must be a nonempty, unpadded string\\")\\n    if value in {\\".\\", \\"..\\"} or any(char in value for char in \\"/\\\\\\\\:\\"):\\n        raise ContractError(\\"identifier must be a single safe path component\\")\\n    if any(unicodedata.category(char) in {\\"Cc\\", \\"Cf\\"} for char in value):\\n        raise ContractError(\\"identifier contains a control character\\")\\n    return value\\n\\n\\ndef _numeric(value: Any) -> bool:\\n    return not isinstance(value, bool) and isinstance(value, (Real, Decimal))\\n\\n\\ndef parse_label(value: Any) -> tuple[float | None, str]:\\n    \\"\\"\\"Classify an observed binary label, an unknown cell, or an invalid cell.\\n\\n    Accepted text is deliberately limited to 0, 1, 0.0 and 1.0. A text token\\n    such as \'NaN\' is invalid; an actual floating NaN represents a missing cell.\\n    \\"\\"\\"\\n    if value is None:\\n        return None, \\"unknown\\"\\n    if isinstance(value, str):\\n        token = value.strip()\\n        if not token:\\n            return None, \\"unknown\\"\\n        if token in {\\"0\\", \\"0.0\\", \\"1\\", \\"1.0\\"}:\\n            return float(token), \\"observed\\"\\n        return None, \\"invalid\\"\\n    if isinstance(value, Real) and not isinstance(value, bool):\\n        try:\\n            if math.isnan(float(value)):\\n                return None, \\"unknown\\"\\n        except (OverflowError, ValueError):\\n            return None, \\"invalid\\"\\n    if _numeric(value):\\n        try:\\n            numeric = float(value)\\n        except (OverflowError, ValueError):\\n            return None, \\"invalid\\"\\n        if math.isfinite(numeric) and numeric in (0.0, 1.0):\\n            return numeric, \\"observed\\"\\n    return None, \\"invalid\\"\\n\\n\\ndef _required(row: Mapping[str, Any], required: tuple[str, ...], index: int) -> None:\\n    if not isinstance(row, Mapping):\\n        raise ContractError(f\\"row {index} must be a mapping\\")\\n    missing = [key for key in required if key not in row]\\n    if missing:\\n        raise ContractError(f\\"row {index} is missing required columns: {\', \'.join(missing)}\\")\\n\\n\\ndef label_vector(row: Mapping[str, Any]) -> dict[str, Any]:\\n    \\"\\"\\"Return twelve ordered values and an explicit observation mask.\\n\\n    Unknown and invalid cells retain None; the state list records the\\n    distinction. Only verified binary observations receive a True mask.\\n    \\"\\"\\"\\n    _required(row, TARGETS, 0)\\n    parsed = [parse_label(row[target]) for target in TARGETS]\\n    return {\\n        \\"targets\\": list(TARGETS),\\n        \\"values\\": [value for value, _ in parsed],\\n        \\"observed_mask\\": [state == \\"observed\\" for _, state in parsed],\\n        \\"states\\": [state for _, state in parsed],\\n    }\\n\\n\\ndef audit_studies(rows: list[dict[str, Any]], split: str) -> dict[str, Any]:\\n    \\"\\"\\"Audit study identities and, for training, labels and report coverage.\\"\\"\\"\\n    if split not in {\\"train\\", \\"test\\"}:\\n        raise ContractError(\\"split must be train or test\\")\\n    if not rows:\\n        raise ContractError(\\"study table must contain at least one row\\")\\n    required = (STUDY_ID, *TARGETS, \\"Report\\") if split == \\"train\\" else (STUDY_ID,)\\n    study_ids: list[str] = []\\n    seen: set[str] = set()\\n    statistics = {\\n        target: {\\"observed\\": 0, \\"positive\\": 0, \\"negative\\": 0, \\"unknown\\": 0, \\"invalid\\": 0}\\n        for target in TARGETS\\n    }\\n    label_states: list[dict[str, tuple[float | None, str]]] = []\\n    report_summary = {\\"present\\": 0, \\"empty_or_missing\\": 0, \\"invalid\\": 0}\\n    for index, row in enumerate(rows):\\n        _required(row, required, index)\\n        uid = validate_identifier(row[STUDY_ID])\\n        if uid in seen:\\n            raise ContractError(f\\"duplicate study identifier at row {index}\\")\\n        seen.add(uid)\\n        study_ids.append(uid)\\n        if split == \\"test\\":\\n            continue\\n        states = {target: parse_label(row[target]) for target in TARGETS}\\n        label_states.append(states)\\n        for target, (label, state) in states.items():\\n            statistics[target][state] += 1\\n            if state == \\"observed\\":\\n                statistics[target][\\"positive\\" if label == 1.0 else \\"negative\\"] += 1\\n        report = row[\\"Report\\"]\\n        if report is None or (isinstance(report, str) and not report.strip()):\\n            report_summary[\\"empty_or_missing\\"] += 1\\n        elif isinstance(report, str):\\n            report_summary[\\"present\\"] += 1\\n        else:\\n            report_summary[\\"invalid\\"] += 1\\n    result: dict[str, Any] = {\\n        \\"status\\": \\"PASS\\", \\"split\\": split, \\"row_count\\": len(rows),\\n        \\"unique_study_count\\": len(seen), \\"study_ids\\": study_ids,\\n    }\\n    if split == \\"test\\":\\n        return result\\n    for stats in statistics.values():\\n        observed = stats[\\"observed\\"]\\n        stats[\\"prevalence\\"] = stats[\\"positive\\"] / observed if observed else None\\n        stats[\\"observation_coverage\\"] = observed / len(rows)\\n    cooccurrence = []\\n    for first_index, first in enumerate(TARGETS):\\n        for second in TARGETS[first_index + 1:]:\\n            jointly_observed = both_positive = 0\\n            for states in label_states:\\n                first_label, first_state = states[first]\\n                second_label, second_state = states[second]\\n                if first_state == second_state == \\"observed\\":\\n                    jointly_observed += 1\\n                    both_positive += first_label == second_label == 1.0\\n            cooccurrence.append({\\n                \\"targets\\": [first, second], \\"jointly_observed\\": jointly_observed,\\n                \\"both_positive\\": both_positive,\\n                \\"both_positive_fraction\\": both_positive / jointly_observed if jointly_observed else None,\\n            })\\n    result.update({\\n        \\"target_statistics\\": statistics, \\"report_summary\\": report_summary,\\n        \\"observation_coverage\\": sum(stats[\\"observed\\"] for stats in statistics.values()) / (len(rows) * len(TARGETS)),\\n        \\"jointly_observed_cooccurrence\\": cooccurrence,\\n    })\\n    if any(stats[\\"invalid\\"] for stats in statistics.values()) or report_summary[\\"invalid\\"]:\\n        result[\\"status\\"] = \\"INVALID\\"\\n    return result\\n\\n\\ndef _metadata_state(value: Any) -> str:\\n    if value is None or (isinstance(value, str) and not value.strip()):\\n        return \\"unknown\\"\\n    if isinstance(value, float):\\n        if math.isnan(float(value)):\\n            return \\"unknown\\"\\n        if not math.isfinite(float(value)):\\n            return \\"invalid\\"\\n    if isinstance(value, (str, bool, int, float)):\\n        return \\"observed\\"\\n    return \\"invalid\\"\\n\\n\\ndef audit_series(rows: list[dict[str, Any]], study_ids: set[str]) -> dict[str, Any]:\\n    \\"\\"\\"Check series uniqueness, study references and exact metadata coverage.\\n\\n    Category vocabularies must be established from real data. No plane or flag\\n    value is coerced here; typed value counts retain the observed representation.\\n    \\"\\"\\"\\n    if not rows:\\n        raise ContractError(\\"series table must contain at least one row\\")\\n    validated_studies = {validate_identifier(uid) for uid in study_ids}\\n    seen: set[str] = set()\\n    referenced: set[str] = set()\\n    metadata_fields = SERIES_COLUMNS[2:]\\n    statistics = {field: {\\"observed\\": 0, \\"unknown\\": 0, \\"invalid\\": 0} for field in metadata_fields}\\n    counts: dict[str, Counter[tuple[str, Any]]] = {field: Counter() for field in metadata_fields}\\n    series_ids: list[str] = []\\n    for index, row in enumerate(rows):\\n        _required(row, SERIES_COLUMNS, index)\\n        study = validate_identifier(row[STUDY_ID])\\n        series = validate_identifier(row[SERIES_ID])\\n        if study not in validated_studies:\\n            raise ContractError(f\\"series row {index} references an absent study\\")\\n        if series in seen:\\n            raise ContractError(f\\"duplicate series identifier at row {index}\\")\\n        seen.add(series)\\n        series_ids.append(series)\\n        referenced.add(study)\\n        for field in metadata_fields:\\n            value = row[field]\\n            state = _metadata_state(value)\\n            statistics[field][state] += 1\\n            if state == \\"observed\\":\\n                counts[field][(type(value).__name__, value)] += 1\\n    for field in metadata_fields:\\n        statistics[field][\\"value_counts\\"] = [\\n            {\\"type\\": value_type, \\"value\\": value, \\"count\\": count}\\n            for (value_type, value), count in sorted(counts[field].items(), key=lambda item: (item[0][0], str(item[0][1])))\\n        ]\\n    return {\\n        \\"status\\": \\"INVALID\\" if any(stats[\\"invalid\\"] for stats in statistics.values()) else \\"PASS\\",\\n        \\"row_count\\": len(rows), \\"unique_series_count\\": len(seen),\\n        \\"study_count_with_series\\": len(referenced), \\"study_count_without_series\\": len(validated_studies - referenced),\\n        \\"series_ids\\": series_ids, \\"metadata_statistics\\": statistics,\\n    }\\n","inputs":"\\"\\"\\"Bounded read-only discovery and CSV reads with explicit incomplete states.\\"\\"\\"\\n\\nimport csv\\nfrom datetime import datetime, timezone\\nimport hashlib\\nimport io\\nimport os\\nfrom pathlib import Path\\nimport stat\\n\\nfrom .config import AuditConfig\\n\\n\\nMANIFEST_NAMES = (\\"train.csv\\", \\"train_series.csv\\", \\"test.csv\\", \\"test_series.csv\\", \\"sample_submission.csv\\")\\nIMAGE_DIRECTORIES = {\\"train_series\\", \\"test_series\\", \\"train_images\\", \\"test_images\\"}\\nPROTECTED_COMPONENTS = {\\".git\\", \\".aws\\", \\".kaggle\\", \\".codex\\", \\".venv\\", \\"venv\\", \\"node_modules\\", \\"credentials\\", \\"access_token\\", \\"kaggle.json\\"}\\n\\n\\nclass InputError(ValueError):\\n    \\"\\"\\"Input contracts or safe-path requirements were not satisfied.\\"\\"\\"\\n\\n\\ndef safe_absolute(path: str | Path) -> Path:\\n    \\"\\"\\"Reject links/reparse ancestors rather than silently resolving through them.\\"\\"\\"\\n    absolute = Path(os.path.abspath(path))\\n    if any(part.casefold() in PROTECTED_COMPONENTS or part.casefold().startswith(\\".env\\") for part in absolute.parts):\\n        raise InputError(\\"Protected directory is not a permitted input or output path\\")\\n    for ancestor in (absolute, *absolute.parents):\\n        try:\\n            info = ancestor.lstat()\\n        except FileNotFoundError:\\n            continue\\n        if stat.S_ISLNK(info.st_mode) or getattr(info, \\"st_file_attributes\\", 0) & 0x400:\\n            raise InputError(\\"Symbolic links and reparse points are not permitted\\")\\n    return absolute\\n\\n\\ndef discover_inputs(config: AuditConfig) -> dict:\\n    result = {\\n        \\"status\\": \\"INPUTS_NOT_LOCATED\\", \\"input_base\\": None, \\"selected_root\\": None,\\n        \\"directories_checked\\": 0, \\"entries_checked\\": 0, \\"scan_complete\\": True,\\n        \\"limit_reached\\": False, \\"errors\\": [], \\"error_count\\": 0,\\n        \\"skipped_links\\": [], \\"skipped_link_count\\": 0, \\"candidates\\": [],\\n        \\"csv_contents_read\\": False, \\"dicom_contents_read\\": False,\\n        \\"dataset_identity_verified\\": False,\\n    }\\n\\n    def issue(path, error):\\n        result[\\"error_count\\"] += 1\\n        if len(result[\\"errors\\"]) < 32:\\n            result[\\"errors\\"].append({\\"path\\": str(path), \\"type\\": type(error).__name__, \\"errno\\": getattr(error, \\"errno\\", None)})\\n\\n    try:\\n        base = safe_absolute(config.input_base)\\n        result[\\"input_base\\"] = str(base)\\n        start = safe_absolute(config.data_root) if config.data_root else base\\n        if not start.is_relative_to(base):\\n            raise InputError(\\"Explicit data_root must be within input_base\\")\\n        info = start.lstat()\\n        if not stat.S_ISDIR(info.st_mode):\\n            raise InputError(\\"Input root is not a directory\\")\\n    except FileNotFoundError:\\n        result[\\"status\\"] = \\"INPUT_ROOT_MISSING\\" if config.data_root else \\"INPUT_BASE_MISSING\\"\\n        return result\\n    except (OSError, InputError) as error:\\n        issue(config.input_base, error)\\n        result.update(status=\\"INPUT_PATH_INVALID\\", scan_complete=False)\\n        return result\\n\\n    def checked_stat(path):\\n        try:\\n            info = path.lstat()\\n            if stat.S_ISLNK(info.st_mode) or getattr(info, \\"st_file_attributes\\", 0) & 0x400:\\n                result[\\"skipped_link_count\\"] += 1\\n                if len(result[\\"skipped_links\\"]) < 32:\\n                    result[\\"skipped_links\\"].append(str(path))\\n                return None\\n            if not path.resolve().is_relative_to(base):\\n                raise InputError(\\"Discovered path escaped input_base\\")\\n            return info\\n        except FileNotFoundError:\\n            return None\\n        except (OSError, InputError) as error:\\n            issue(path, error)\\n            return None\\n\\n    pending = [(start, 0)]\\n    cursor = 0\\n    while cursor < len(pending) and result[\\"directories_checked\\"] < config.max_directories:\\n        directory, depth = pending[cursor]\\n        cursor += 1\\n        result[\\"directories_checked\\"] += 1\\n        files = []\\n        for name in MANIFEST_NAMES:\\n            path = directory / name\\n            info = checked_stat(path)\\n            if info is not None and stat.S_ISREG(info.st_mode):\\n                files.append({\\"name\\": name, \\"path\\": str(path), \\"size_bytes\\": info.st_size,\\n                              \\"modified_utc\\": datetime.fromtimestamp(info.st_mtime, timezone.utc).isoformat(), \\"sha256\\": None})\\n        if files:\\n            present = {item[\\"name\\"] for item in files}\\n            images = {}\\n            for name in sorted(IMAGE_DIRECTORIES):\\n                path = directory / name\\n                info = checked_stat(path)\\n                if info is not None and stat.S_ISDIR(info.st_mode):\\n                    images[name] = str(path)\\n            result[\\"candidates\\"].append({\\"root\\": str(directory), \\"files\\": files,\\n                                         \\"missing\\": [name for name in MANIFEST_NAMES if name not in present],\\n                                         \\"image_directory_candidates\\": images, \\"dicom_file_count\\": None})\\n        if config.data_root is not None or depth >= config.max_depth:\\n            continue\\n        try:\\n            with os.scandir(directory) as entries:\\n                for entry in entries:\\n                    if result[\\"entries_checked\\"] >= config.max_entries:\\n                        result[\\"limit_reached\\"] = True\\n                        break\\n                    result[\\"entries_checked\\"] += 1\\n                    if entry.name.startswith(\\".\\") or entry.name in IMAGE_DIRECTORIES or entry.name.casefold() in PROTECTED_COMPONENTS:\\n                        continue\\n                    child = directory / entry.name\\n                    info = checked_stat(child)\\n                    if info is not None and stat.S_ISDIR(info.st_mode):\\n                        pending.append((child, depth + 1))\\n        except OSError as error:\\n            issue(directory, error)\\n        if result[\\"limit_reached\\"]:\\n            break\\n    result[\\"scan_complete\\"] = cursor == len(pending) and not result[\\"limit_reached\\"] and not result[\\"error_count\\"]\\n    result[\\"candidates\\"].sort(key=lambda item: item[\\"root\\"])\\n    complete = [item for item in result[\\"candidates\\"] if not item[\\"missing\\"]]\\n    if len(complete) > 1:\\n        result[\\"status\\"] = \\"AMBIGUOUS_INPUT_ROOTS\\"\\n    elif not result[\\"scan_complete\\"]:\\n        result[\\"status\\"] = \\"DISCOVERY_INCOMPLETE\\"\\n    elif complete:\\n        result.update(status=\\"MANIFESTS_LOCATED\\", selected_root=complete[0][\\"root\\"])\\n    elif result[\\"candidates\\"]:\\n        result[\\"status\\"] = \\"INCOMPLETE_MANIFESTS\\"\\n    return result\\n\\n\\ndef read_csv_manifest(path: str | Path, config: AuditConfig) -> tuple[list[str], list[dict], str]:\\n    source = safe_absolute(path)\\n    base = safe_absolute(config.input_base)\\n    if not source.is_relative_to(base) or source.name not in MANIFEST_NAMES:\\n        raise InputError(\\"CSV source must be a named competition manifest within input_base\\")\\n    if not stat.S_ISREG(source.lstat().st_mode):\\n        raise InputError(\\"CSV source must be a regular file\\")\\n    with source.open(\\"rb\\") as stream:\\n        if not stat.S_ISREG(os.fstat(stream.fileno()).st_mode):\\n            raise InputError(\\"CSV source must be a regular file\\")\\n        raw = stream.read(config.max_file_bytes + 1)\\n    if len(raw) > config.max_file_bytes:\\n        raise InputError(\\"CSV_FILE_LIMIT_EXCEEDED\\")\\n    try:\\n        reader = csv.DictReader(io.StringIO(raw.decode(\\"utf-8-sig\\"), newline=\\"\\"), strict=True)\\n        columns = reader.fieldnames\\n        if not columns or any(not name for name in columns) or len(set(columns)) != len(columns):\\n            raise InputError(\\"CSV_HEADERS_MISSING_OR_DUPLICATED\\")\\n        rows = []\\n        for number, row in enumerate(reader, start=1):\\n            if number > config.max_csv_rows:\\n                raise InputError(\\"CSV_ROW_LIMIT_EXCEEDED\\")\\n            if None in row or any(value is None for value in row.values()):\\n                raise InputError(\\"CSV_ROW_WIDTH_INVALID\\")\\n            rows.append(row)\\n    except (UnicodeError, csv.Error) as error:\\n        raise InputError(\\"CSV_ENCODING_OR_FORMAT_INVALID\\") from error\\n    return columns, rows, hashlib.sha256(raw).hexdigest()\\n","metrics":"\\"\\"\\"Observed-label ROC AUC with explicit support and undefined macro metrics.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nfrom collections.abc import Iterable, Mapping, Sequence\\nfrom typing import Any\\n\\nfrom .contracts import ContractError, STUDY_ID, TARGETS, _numeric, parse_label, validate_identifier\\nfrom .submission import validate_submission\\n\\n\\ndef binary_auc(labels: Iterable[Any], scores: Iterable[Any]) -> float | None:\\n    \\"\\"\\"Compute rank AUC, averaging tied ranks; return None for missing classes.\\"\\"\\"\\n    observed_labels = list(labels)\\n    observed_scores = list(scores)\\n    if len(observed_labels) != len(observed_scores):\\n        raise ContractError(\\"AUC labels and scores must have identical lengths\\")\\n    pairs = []\\n    for label, score in zip(observed_labels, observed_scores):\\n        if not _numeric(label) or parse_label(label)[1] != \\"observed\\":\\n            raise ContractError(\\"AUC requires observed numeric binary labels\\")\\n        if not _numeric(score):\\n            raise ContractError(\\"AUC requires numeric scores\\")\\n        try:\\n            numeric_score = float(score)\\n        except (OverflowError, ValueError) as error:\\n            raise ContractError(\\"AUC requires finite scores\\") from error\\n        if not math.isfinite(numeric_score):\\n            raise ContractError(\\"AUC requires finite scores\\")\\n        pairs.append((numeric_score, float(label)))\\n    positive_count = sum(label == 1.0 for _, label in pairs)\\n    negative_count = len(pairs) - positive_count\\n    if not positive_count or not negative_count:\\n        return None\\n    pairs.sort(key=lambda pair: pair[0])\\n    positive_rank_sum = 0.0\\n    start = 0\\n    while start < len(pairs):\\n        end = start + 1\\n        while end < len(pairs) and pairs[end][0] == pairs[start][0]:\\n            end += 1\\n        mean_rank = (start + 1 + end) / 2.0\\n        positive_rank_sum += mean_rank * sum(label == 1.0 for _, label in pairs[start:end])\\n        start = end\\n    return (positive_rank_sum - positive_count * (positive_count + 1) / 2.0) / (positive_count * negative_count)\\n\\n\\ndef evaluate_predictions(\\n    label_rows: Sequence[Mapping[str, Any]],\\n    prediction_rows: Sequence[Mapping[str, Any]],\\n) -> dict[str, Any]:\\n    \\"\\"\\"Match studies exactly and evaluate each target on observed labels only.\\"\\"\\"\\n    if not label_rows:\\n        raise ContractError(\\"reference label table must not be empty\\")\\n    references: dict[str, Mapping[str, Any]] = {}\\n    for index, row in enumerate(label_rows):\\n        if not isinstance(row, Mapping) or not set((STUDY_ID, *TARGETS)).issubset(row):\\n            raise ContractError(f\\"reference row {index} is missing required cells\\")\\n        uid = validate_identifier(row[STUDY_ID])\\n        if uid in references:\\n            raise ContractError(f\\"duplicate reference identifier at row {index}\\")\\n        references[uid] = row\\n    validate_submission(prediction_rows, references)\\n    predictions = {row[STUDY_ID]: row for row in prediction_rows}\\n    per_target = {}\\n    for target in TARGETS:\\n        labels = []\\n        scores = []\\n        unknown_count = 0\\n        for uid, row in references.items():\\n            label, state = parse_label(row[target])\\n            if state == \\"invalid\\":\\n                raise ContractError(f\\"reference contains an invalid label for {target}\\")\\n            if state == \\"unknown\\":\\n                unknown_count += 1\\n                continue\\n            labels.append(label)\\n            scores.append(predictions[uid][target])\\n        positives = sum(label == 1.0 for label in labels)\\n        per_target[target] = {\\n            \\"observed\\": len(labels), \\"positive\\": positives,\\n            \\"negative\\": len(labels) - positives, \\"unknown\\": unknown_count,\\n            \\"auc\\": binary_auc(labels, scores),\\n        }\\n    defined = [stats[\\"auc\\"] for stats in per_target.values() if stats[\\"auc\\"] is not None]\\n    complete = len(defined) == len(TARGETS)\\n    return {\\n        \\"status\\": \\"FULL_METRICS_AVAILABLE\\" if complete else \\"INCOMPLETE_METRICS\\",\\n        \\"row_count\\": len(references), \\"per_target\\": per_target,\\n        \\"macro_auc\\": sum(defined) / len(TARGETS) if complete else None,\\n        \\"diagnostic_defined_target_macro_auc\\": sum(defined) / len(defined) if defined else None,\\n        \\"defined_target_count\\": len(defined), \\"required_target_count\\": len(TARGETS),\\n    }\\n","submission":"\\"\\"\\"Validate the exact submission contract and write without overwriting files.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport csv\\nimport math\\nfrom collections.abc import Iterable, Mapping, Sequence\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nfrom .contracts import ContractError, STUDY_ID, TARGETS, _numeric, validate_identifier\\nfrom .inputs import safe_absolute\\n\\nSUBMISSION_COLUMNS = (STUDY_ID, *TARGETS)\\n\\n\\ndef validate_submission(\\n    rows: Sequence[Mapping[str, Any]],\\n    test_ids: Iterable[str],\\n    columns: Sequence[str] | None = None,\\n) -> dict[str, Any]:\\n    \\"\\"\\"Require each test study once and twelve finite probabilities in order.\\"\\"\\"\\n    expected_ids = [validate_identifier(uid) for uid in test_ids]\\n    if not expected_ids:\\n        raise ContractError(\\"test identifier set must not be empty\\")\\n    if len(set(expected_ids)) != len(expected_ids):\\n        raise ContractError(\\"test identifier list contains duplicates\\")\\n    if not rows:\\n        raise ContractError(\\"submission must contain at least one row\\")\\n    if not isinstance(rows[0], Mapping):\\n        raise ContractError(\\"submission row 0 must be a mapping\\")\\n    actual_columns = tuple(columns) if columns is not None else tuple(rows[0])\\n    if actual_columns != SUBMISSION_COLUMNS:\\n        raise ContractError(\\"submission columns must exactly match the identifier and target order\\")\\n    observed_ids: set[str] = set()\\n    for index, row in enumerate(rows):\\n        if not isinstance(row, Mapping) or set(row) != set(SUBMISSION_COLUMNS):\\n            raise ContractError(f\\"submission row {index} has missing or extra cells\\")\\n        uid = validate_identifier(row[STUDY_ID])\\n        if uid in observed_ids:\\n            raise ContractError(f\\"duplicate submission identifier at row {index}\\")\\n        observed_ids.add(uid)\\n        for target in TARGETS:\\n            value = row[target]\\n            if not _numeric(value):\\n                raise ContractError(f\\"submission row {index} has a nonnumeric probability for {target}\\")\\n            try:\\n                probability = float(value)\\n            except (ValueError, OverflowError) as error:\\n                raise ContractError(f\\"submission row {index} has an invalid probability for {target}\\") from error\\n            if not math.isfinite(probability) or not 0.0 <= probability <= 1.0:\\n                raise ContractError(f\\"submission row {index} has an invalid probability for {target}\\")\\n    if observed_ids != set(expected_ids):\\n        raise ContractError(\\"submission identifier set differs from the exact test identifier set\\")\\n    return {\\n        \\"status\\": \\"PASS\\", \\"row_count\\": len(rows), \\"unique_study_count\\": len(observed_ids),\\n        \\"columns\\": list(SUBMISSION_COLUMNS), \\"probabilities_per_row\\": len(TARGETS),\\n    }\\n\\n\\ndef write_submission(\\n    rows: Sequence[Mapping[str, Any]],\\n    test_ids: Iterable[str],\\n    path: str | Path,\\n    columns: Sequence[str] = SUBMISSION_COLUMNS,\\n) -> dict[str, Any]:\\n    \\"\\"\\"Validate, reject protected/link paths, and refuse any existing output.\\"\\"\\"\\n    validation = validate_submission(rows, test_ids, columns)\\n    destination = safe_absolute(path)\\n    with destination.open(\\"x\\", newline=\\"\\", encoding=\\"utf-8\\") as stream:\\n        writer = csv.DictWriter(stream, fieldnames=list(columns))\\n        writer.writeheader()\\n        writer.writerows(rows)\\n    return {**validation, \\"path\\": str(destination), \\"size_bytes\\": destination.stat().st_size}\\n"}}')
_RSNA_PACKAGE = 'rsna_knee_bundle'
for _module, _source in _RSNA_BUNDLE["sources"].items():
    _actual = hashlib.sha256(_source.encode("utf-8")).hexdigest()
    if _actual != _RSNA_BUNDLE["sha256"][_module + ".py"]:
        raise RuntimeError("Embedded source checksum mismatch: " + _module)
_RSNA_SOURCE_IDENTITY = {
    "kind": "embedded-notebook",
    "bundle_sha256": hashlib.sha256(json.dumps(_RSNA_BUNDLE, sort_keys=True, ensure_ascii=True, separators=(",", ":")).encode("utf-8")).hexdigest(),
    "source_sha256": dict(_RSNA_BUNDLE["sha256"]),
}

class _RsnaBundleLoader(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    package_name = _RSNA_PACKAGE

    def __init__(self, sources, fingerprint):
        # Bind this loader to its own bundle, not mutable notebook globals.
        self.sources = dict(sources)
        self.fingerprint = fingerprint

    def find_spec(self, fullname, path=None, target=None):
        if fullname == self.package_name:
            return importlib.util.spec_from_loader(fullname, self, is_package=True)
        prefix = self.package_name + "."
        if fullname.startswith(prefix):
            name = fullname[len(prefix):]
            if name in self.sources and name != "__init__":
                return importlib.util.spec_from_loader(fullname, self)
        return None

    def create_module(self, spec):
        return None

    def exec_module(self, module):
        name = "__init__" if module.__name__ == self.package_name else module.__name__.split(".")[-1]
        module.__file__ = "<embedded:" + name + ".py>"
        module.__rsna_bundle_sha256__ = self.fingerprint
        code = compile(self.sources[name], module.__file__, "exec")
        exec(code, module.__dict__)

# Preserve class identity when re-running an identical, verified bundle.
# Changed sources are reloaded and require the configuration cell to run again.
_rsna_fingerprint = _RSNA_SOURCE_IDENTITY["bundle_sha256"]
_rsna_cached = {
    name: module for name, module in sys.modules.items()
    if name == _RSNA_PACKAGE or name.startswith(_RSNA_PACKAGE + ".")
}
_rsna_same_bundle = (
    _RSNA_PACKAGE in _rsna_cached
    and all(getattr(module, "__rsna_bundle_sha256__", None) == _rsna_fingerprint
            for module in _rsna_cached.values())
)
if not _rsna_same_bundle:
    for _name in _rsna_cached:
        del sys.modules[_name]
    sys.meta_path[:] = [finder for finder in sys.meta_path if getattr(finder, "package_name", None) != _RSNA_PACKAGE]
    sys.meta_path.insert(0, _RsnaBundleLoader(_RSNA_BUNDLE["sources"], _rsna_fingerprint))
from rsna_knee_bundle.config import AuditConfig
from rsna_knee_bundle.audit import run_preflight, write_report


In [ ]:
# Generated configuration. Use the project builder to change settings.
CONFIG = {'input_base': '/kaggle/input',
 'inspect_csv': False,
 'output_dir': '/kaggle/working/rsna_audit'}
audit_config = AuditConfig.from_dict(CONFIG)


In [ ]:
if 'audit_config' not in globals() or not isinstance(audit_config, AuditConfig):
    raise RuntimeError('Missing or stale AuditConfig: rerun the configuration cell before the audit, or use Run All.')
audit_report = run_preflight(audit_config, source_identity=_RSNA_SOURCE_IDENTITY)
print(json.dumps(audit_report, sort_keys=True, indent=2))


In [ ]:
if 'audit_report' not in globals() or 'audit_config' not in globals() or not isinstance(audit_config, AuditConfig):
    raise RuntimeError('Run the configuration and audit cells first, or use Run All.')
if audit_report.get('source_identity') != _RSNA_SOURCE_IDENTITY or audit_report.get('configuration') != audit_config.to_dict():
    raise RuntimeError('The audit report is stale; run the configuration and audit cells again, or use Run All.')
report_path = write_report(audit_report, audit_config)
print(str(report_path))
